# `baidu-asr` 全接口调用演示 (Demo Notebook)

本 Notebook 演示 **零第三方依赖、高性能百度输入法长语音与音视频转文字 SDK (`baidu-asr`)** 的全部调用方式：

1. **环境检查与公开符号验证 (`dependencies = []`)**
2. **方式一：一行代码极速转写 + 可选智能标点整理 (`transcribe`)**
3. **方式二：多格式一键导出（`Text` / `SRT` 字幕 / `WebVTT` 字幕 / `Markdown` / `JSON`）**
4. **方式三：长音频自适应 VAD 静音切分 + 8 线程并发极速转写 (`BaiduASR.transcribe(workers=8)`)**
5. **方式四：生成器流式转写 (`transcribe_stream` — Pull Stream 边解码边出字)**
6. **方式五：协议级实时推流会话 (`asr.create_stream` — Push Stream 麦克风/WebSocket 实时音频流)**
7. **方式六：向后兼容异步 `recognize` 与 `load_audio` 及 `atranscribe_stream`**
8. **方式七：底层音频与 32kbps CBR `restricted_lowdelay` Opus 编码器直接调用**
9. **方式八：CLI 命令行调用演示 (`python -m baidu_asr`)**

In [1]:
import os
import sys

sys.path.insert(0, os.path.abspath("src"))

import baidu_asr
from baidu_asr import (
    BaiduASR,
    BaiduStreamSession,
    BaiduOpusEncoder,
    transcribe,
    transcribe_stream,
    recognize,
    load_audio,
    organize_text,
    opus_packets,
    encode_pcm_to_baidu_ws_frames,
    compute_rms,
    split_pcm_by_silence,
)

print("baidu_asr 版本:", baidu_asr.__version__)
print("公开导出符号:", baidu_asr.__all__)

baidu_asr 版本: 1.0.0
公开导出符号: ['BaiduASR', 'BaiduStreamSession', 'transcribe', 'transcribe_stream', 'recognize', 'load_audio', 'organize_text', 'Segment', 'StreamEvent', 'TranscriptResult', 'BaiduOpusEncoder', 'opus_packets', 'encode_pcm_to_baidu_ws_frames', 'compute_rms', 'split_pcm_by_silence', 'iter_pcm16k_chunks', 'load_pcm16k']


---
## 方式一：一行代码极速同步转写 + 智能标点整理 (`transcribe`)

In [2]:
res = transcribe("examples/asr_example_zh.wav", organize=True)

print("智能整理成稿:", res.text)
print("百度 ASR 原文:", res.raw_text)
print("音频时长:", f"{res.duration}s | 转写耗时: {res.elapsed_sec}s")
print("分段明细:", res.segments)

智能整理成稿: 欢迎大家来体验达摩院推出的语音识别模型。
百度 ASR 原文: 欢迎大家来体验达摩院推出的语音识别模型。
音频时长: 5.55s | 转写耗时: 1.44s
分段明细: [Segment(index=1, start=0.0, end=5.55, text='欢迎大家来体验达摩院推出的语音识别模型', rms=4000.4, confidence=99, speak_speed=3.42, voice_power=-21.92, alternatives=['欢迎大家来体验达摩院推出的语音识别模型'])]


---
## 方式二：多格式一键导出（`SRT` / `VTT` / `Markdown` / `JSON`）

In [3]:
print("=== [1] SRT 字幕格式 ===")
print(res.to_srt())

print("=== [2] WebVTT 字幕格式 ===")
print(res.to_vtt())

print("=== [3] Markdown 报告格式 ===")
print(res.to_markdown(title="达摩院语音示例百度 ASR 转写稿"))

print("=== [4] JSON 结构化格式 ===")
print(res.to_json())

=== [1] SRT 字幕格式 ===
1
00:00:00,000 --> 00:00:05,550
欢迎大家来体验达摩院推出的语音识别模型

=== [2] WebVTT 字幕格式 ===
WEBVTT

00:00:00.000 --> 00:00:05.550
欢迎大家来体验达摩院推出的语音识别模型

=== [3] Markdown 报告格式 ===
# 达摩院语音示例百度 ASR 转写稿

- **音频来源**: `examples/asr_example_zh.wav`
- **音频时长**: `5.5s`
- **转写耗时**: `1.44s`
- **实时倍率**: `3.9x`
- **有效片段**: `1 / 1` 段

## 一、 智能整理成稿

欢迎大家来体验达摩院推出的语音识别模型。

## 二、 百度 ASR 原始识别文本

欢迎大家来体验达摩院推出的语音识别模型。

## 三、 分段时间轴明细

| 序号 | 时间轴 | 置信度 | 语速 | 识别文本 |
| :---: | :---: | :---: | :---: | :--- |
| 01 | `00:00.0 - 00:05.5` | `99%` | `3.42` | 欢迎大家来体验达摩院推出的语音识别模型 |

=== [4] JSON 结构化格式 ===
{
  "source": "examples/asr_example_zh.wav",
  "duration": 5.55,
  "elapsed_sec": 1.44,
  "text": "欢迎大家来体验达摩院推出的语音识别模型。",
  "raw_text": "欢迎大家来体验达摩院推出的语音识别模型。",
  "organized_text": "欢迎大家来体验达摩院推出的语音识别模型。",
  "segments": [
    {
      "index": 1,
      "start": 0.0,
      "end": 5.55,
      "text": "欢迎大家来体验达摩院推出的语音识别模型",
      "rms": 4000.4,
      "confidence": 99,
      "speak_speed": 3.42,
      "voice_power": -2

---
## 方式三：长音频自适应 VAD 静音切分 + 8 线程并发极速转写 (`BaiduASR.transcribe(workers=8)`)

In [4]:
with BaiduASR() as asr:
    pcm1 = asr.load_pcm16k("examples/asr_example_zh.wav")
    pcm2 = asr.load_pcm16k("examples/asr_example.wav")
    silence_1s = b"\x00\x00" * 16000
    long_pcm = pcm1 + silence_1s + pcm2 + silence_1s + pcm1

    long_res = asr.transcribe(
        long_pcm,
        min_segment_sec=4.0,
        max_segment_sec=7.0,
        workers=8,
        organize=True,
        on_segment=lambda seg: print(f"  -> [实时回调 #{seg.index:02d}] [{seg.timestamp}] (conf={seg.confidence}%, speed={seg.speak_speed}) {seg.text}"),
    )

print(f"\n并发转写完成！总时长: {long_res.duration}s | 实际耗时: {long_res.elapsed_sec}s ({long_res.duration/max(0.01, long_res.elapsed_sec):.1f}x 实时)")
print("百度 ASR 原文:", long_res.raw_text)
print("智能整理成稿:\n" + long_res.text)

  -> [实时回调 #02] [00:05.6 - 00:09.6] (conf=99%, speed=2.0) 每一天都要快乐哦
  -> [实时回调 #03] [00:09.6 - 00:15.5] (conf=99%, speed=3.19) 欢迎大家来体验达摩院推出的语音识别模型
  -> [实时回调 #01] [00:00.0 - 00:05.6] (conf=99%, speed=3.39) 欢迎大家来体验达摩院推出的语音识别模型

并发转写完成！总时长: 15.53s | 实际耗时: 1.22s (12.7x 实时)
百度 ASR 原文: 欢迎大家来体验达摩院推出的语音识别模型，每一天都要快乐哦，欢迎大家来体验达摩院推出的语音识别模型。
智能整理成稿:
欢迎大家来体验达摩院推出的语音识别模型。每一天都要快乐哦。

欢迎大家来体验达摩院推出的语音识别模型。


---
## 方式四：生成器流式转写接口 (`transcribe_stream` — Pull Stream)

In [5]:
for ev in transcribe_stream("examples/asr_example_zh.wav", chunk_sec=0.8, include_partial=True):
    status = "FINAL  (定稿)" if ev.is_final else "PARTIAL(流式)"
    print(f"[{status}] seg=#{ev.segment_index} seq={ev.seq:02d} [{ev.timestamp}] text={ev.text!r} | delta={ev.delta!r}")

[PARTIAL(流式)] seg=#1 seq=06 [00:00.0 - 00:04.8] text='欢迎大家来' | delta='欢迎大家来'
[PARTIAL(流式)] seg=#1 seq=07 [00:00.0 - 00:05.5] text='欢迎大家来体' | delta='体'
[FINAL  (定稿)] seg=#1 seq=08 [00:00.0 - 00:05.5] text='欢迎大家来体验达摩院推出的语音识别模型' | delta='验达摩院推出的语音识别模型'


---
## 方式五：协议级实时推流会话 (`asr.create_stream` — Push Stream)

In [6]:
with BaiduASR() as asr:
    raw_pcm = asr.load_pcm16k("examples/asr_example.wav")
    step_bytes = 6400  # 200ms @ 16kHz 16-bit mono
    with asr.create_stream(chunk_sec=0.6, source_name="mic_simulation") as stream:
        for offset in range(0, len(raw_pcm), step_bytes):
            for ev in stream.feed(raw_pcm[offset : offset + step_bytes]):
                print(f"[Push Event: {ev.type.upper():7s}] [{ev.timestamp}] {ev.text} (delta={ev.delta!r})")
    print("\n推流会话最终文本:", stream.result.text)


推流会话最终文本: 每一天都要快乐哦。


---
## 方式六：向后兼容异步 `recognize` / `load_audio` 与 `atranscribe_stream`

In [7]:
import asyncio

async def run_async_demo():
    pcm = load_audio("examples/asr_example_zh.wav", 16000)
    partials = []
    txt = await recognize(pcm, on_partial=lambda t: partials.append(t))
    print("recognize final:", txt, "| partial 次数:", len(partials))

    with BaiduASR() as asr:
        async for ev in asr.atranscribe_stream("examples/asr_example.wav", chunk_sec=0.8):
            if ev.is_final:
                print("Async Final Event:", ev.to_dict())

asyncio.run(run_async_demo())

recognize final: 欢迎大家来体验达摩院推出的语音识别模型 | partial 次数: 5
Async Final Event: {'type': 'final', 'segment_index': 1, 'seq': 5, 'start': 0.0, 'end': 2.43, 'text': '每一天都要快乐哦', 'delta': '都要快乐哦', 'full_text': '每一天都要快乐哦', 'is_final': True}


---
## 方式七：底层 32kbps CBR `restricted_lowdelay` Opus 编码器直接调用

In [8]:
with BaiduASR() as asr:
    pcm_data = asr.load_pcm16k("examples/asr_example_zh.wav")

pkts = opus_packets(pcm_data)
ws_blob = encode_pcm_to_baidu_ws_frames(pcm_data)
print(f"PCM 长度: {len(pcm_data)} 字节 -> {len(pkts)} 个 20ms CBR 80B Opus 包 | 预打包 WS 帧流: {len(ws_blob)} 字节")

PCM 长度: 177494 字节 -> 278 个 20ms CBR 80B Opus 包 | 预打包 WS 帧流: 26706 字节


---
## 方式八：CLI 命令行调用演示 (`python -m baidu_asr`)

In [9]:
import subprocess

cmd = [sys.executable, "-m", "baidu_asr", "examples/asr_example.wav", "-f", "srt"]
env = dict(os.environ, PYTHONPATH=os.path.abspath("src"), PYTHONIOENCODING="utf-8")
proc = subprocess.run(cmd, stdin=subprocess.DEVNULL, capture_output=True, text=True, encoding="utf-8", env=env)
print(proc.stdout)

[01] [00:00.0 - 00:02.4] 每一天都要快乐哦
1
00:00:00,000 --> 00:00:02,430
每一天都要快乐哦


